# Data Leakage, Imbalanced Data, and Model Interpretability with SHAP

In this notebook, we build an XGBoost classifier to predict high-value homes using the California Housing dataset. Along the way, we encounter two common pitfalls — **data leakage** and **class imbalance** — and use SHAP (SHapley Additive exPlanations) to diagnose problems and interpret our model's predictions.

We will cover:
1. Building a classification model and using SHAP to investigate suspiciously high performance.
2. Identifying and fixing data leakage.
3. Addressing class imbalance when the model ignores the minority class.
4. Interpreting a trustworthy model with SHAP waterfall, beeswarm, and dependence plots.

*SHAP reference: [A Non-Technical Guide to Interpreting SHAP Analyses](https://www.aidancooper.co.uk/a-non-technical-guide-to-interpreting-shap-analyses/) by Aidan Cooper*

In [ ]:
# If you haven't installed shap yet, uncomment and run the line below:
# !pip install shap

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix)
from xgboost import XGBClassifier
import shap

## **Step 1: Load and Inspect the Data**

The California Housing dataset contains block-level housing statistics from the 1990 U.S. Census. Each row represents a census block group with features describing neighborhood characteristics and a median house value target.

We also include `TaxAssessment`, a property tax assessment score sourced from county records.

**Note:** The original dataset caps home values at \$500,001. We remove these capped observations so that our classification threshold reflects genuine variation in home prices, not an artificial ceiling.

In [ ]:
housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target

# Remove capped values ($500,001 ceiling in the original census data)
df = df[df['MedHouseVal'] < 5.00001].reset_index(drop=True)

# Additional feature: property tax assessment score (from county records)
np.random.seed(42)
df['TaxAssessment'] = df['MedHouseVal'] + np.random.normal(0, 0.05, len(df))

print(f"Dataset shape: {df.shape} (after removing {20640 - len(df)} capped values)")
print(f"\nFeatures:")
for name, desc in zip(list(housing.feature_names) + ['TaxAssessment'],
                      ['Median income (tens of thousands)',
                       'Median house age (years)',
                       'Average rooms per household',
                       'Average bedrooms per household',
                       'Block group population',
                       'Average occupants per household',
                       'Latitude', 'Longitude',
                       'Property tax assessment score']):
    print(f"  {name:15s} — {desc}")

df.head()

## **Step 2: Define the Classification Task**

We convert this into a binary classification problem: **is a home in the top 10% of the market by value?** This is a realistic task — identifying premium properties based on neighborhood characteristics — and it naturally creates an imbalanced dataset.

In [ ]:
threshold = np.percentile(df['MedHouseVal'], 90)
df['HighValue'] = (df['MedHouseVal'] >= threshold).astype(int)

print(f"High-value threshold: ${threshold * 100_000:,.0f}")
print(f"\nClass distribution:")
print(f"  Standard (0):   {(df['HighValue'] == 0).sum():,} ({(df['HighValue'] == 0).mean():.1%})")
print(f"  High Value (1): {(df['HighValue'] == 1).sum():,} ({(df['HighValue'] == 1).mean():.1%})")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
df['HighValue'].value_counts().sort_index().plot(
    kind='bar', color=['steelblue', 'coral'], ax=ax, edgecolor='black'
)
ax.set_xticklabels(['Standard (0)', 'High Value (1)'], rotation=0)
ax.set_title('Class Distribution')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## **Step 3: Train-Test Split**

We split the data into training and testing sets using all 9 available features.

In [ ]:
feature_cols = list(housing.feature_names) + ['TaxAssessment']
X = df[feature_cols]
y = df['HighValue']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]:,} samples")
print(f"Testing set:  {X_test.shape[0]:,} samples")
print(f"Features:     {X_train.shape[1]}")

## **Step 4: Train an XGBoost Model**

We train an XGBoost classifier and evaluate its performance on the test set.

In [ ]:
xgb_model = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                           random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.1%}")
print()
print(classification_report(y_test, y_pred, target_names=['Standard', 'High Value']))

## **Step 5: Introduction to SHAP Analysis**

Our model achieved near-perfect accuracy. To understand *why*, we turn to SHAP (SHapley Additive exPlanations).

In Week 10, we used `feature_importances_` from XGBoost. In Week 12, we examined PCA loadings. These tell us **which** features matter, but not **how** they affect individual predictions. SHAP provides both:

$$\text{prediction} = \text{base value} + \sum(\text{SHAP values})$$

The **base value** is the model's average prediction across the training data. Each **SHAP value** represents how much a single feature pushes a prediction up or down from the base. This is analogous to a starting salary (base value) adjusted by individual factors like education, experience, and location.

In [ ]:
explainer_leaky = shap.TreeExplainer(xgb_model)
shap_values_leaky = explainer_leaky(X_test)

print(f"Base value: {shap_values_leaky.base_values[0]:.4f}")
print(f"  (Positive = leans toward High Value, Negative = leans toward Standard)")
print(f"\nSHAP values computed for {X_test.shape[0]:,} samples across {X_test.shape[1]} features.")

## **Step 6: SHAP Feature Importance**

We begin by examining which features the model relies on most heavily. The bar plot shows the **mean absolute SHAP value** for each feature — a measure of how much each feature contributes to predictions on average.

In [ ]:
mean_shap_leaky = np.abs(shap_values_leaky.values).mean(axis=0)
shap_imp_leaky = pd.Series(mean_shap_leaky, index=feature_cols)

plt.figure(figsize=(10, 6))
shap_imp_leaky.sort_values().plot(kind='barh', color='coral', edgecolor='black')
plt.title('SHAP Feature Importance')
plt.xlabel('Mean |SHAP Value|')
plt.tight_layout()
plt.show()

print(f"TaxAssessment:  mean |SHAP| = {shap_imp_leaky['TaxAssessment']:.3f}")
next_best = shap_imp_leaky.drop('TaxAssessment').idxmax()
print(f"{next_best:15s}: mean |SHAP| = {shap_imp_leaky[next_best]:.3f}")

## **Step 7: Investigating the Anomaly**

`TaxAssessment` dominates every other feature by a wide margin. When a single feature drives nearly all of the model's predictions, we should investigate its relationship to the target variable.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df['MedHouseVal'], df['TaxAssessment'], alpha=0.1, s=5, color='steelblue')
ax.set_xlabel('Median House Value ($100K)')
ax.set_ylabel('Tax Assessment Score')
ax.set_title('TaxAssessment vs. Actual Home Value')
plt.tight_layout()
plt.show()

corr = df['MedHouseVal'].corr(df['TaxAssessment'])
print(f"Correlation between TaxAssessment and MedHouseVal: {corr:.4f}")
print(f"\nTaxAssessment is nearly identical to the home's actual value.")
print(f"Since HighValue is derived from MedHouseVal, TaxAssessment gives")
print(f"the model a direct shortcut to the answer. This is data leakage.")

## **Step 8: What is Data Leakage?**

**Data leakage** occurs when information that would not be available at prediction time leaks into the model during training. There are two common forms:

**Target leakage** happens when a feature is derived from or strongly correlated with the target variable. In our case, `TaxAssessment` is essentially the home value itself — the very quantity our binary target is derived from. The model learned a trivial shortcut instead of genuine housing patterns.

**Preprocessing leakage** happens when transformations (like scaling) are fit on the full dataset before splitting. The scaler's statistics then include information from the test set.

Both forms produce artificially inflated performance that will not hold up in the real world.

## **Step 9: Rebuild Without the Leaky Feature**

We remove `TaxAssessment` from the feature set and retrain the model. The same train-test split is preserved — only the feature set changes.

In [ ]:
X_train_clean = X_train[housing.feature_names]
X_test_clean = X_test[housing.feature_names]

xgb_clean = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                           random_state=42, eval_metric='logloss')
xgb_clean.fit(X_train_clean, y_train)
y_pred_clean = xgb_clean.predict(X_test_clean)

print(f"Clean model accuracy: {accuracy_score(y_test, y_pred_clean):.1%}")

In [ ]:
cm = confusion_matrix(y_test, y_pred_clean)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Standard', 'High Value'],
            yticklabels=['Standard', 'High Value'])
plt.title('Confusion Matrix — Clean Model (Without TaxAssessment)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred_clean,
                            target_names=['Standard', 'High Value']))

## **Step 10: The Accuracy Trap — Imbalanced Data**

Our clean model reaches ~94% accuracy. But a `DummyClassifier` that always predicts "Standard" would achieve ~90% — simply because 90% of homes are standard. Our model's accuracy is only marginally better than doing nothing.

The confusion matrix from Step 9 reveals the real problem: the model **misses a large fraction of high-value homes** (low recall on the minority class). This is the **accuracy trap** — high accuracy can mask poor performance on the class we care about most.

In [ ]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train_clean, y_train)
dummy_acc = accuracy_score(y_test, dummy.predict(X_test_clean))

print(f"Dummy Classifier (always predicts Standard): {dummy_acc:.1%}")
print(f"Clean XGBoost:                               {accuracy_score(y_test, y_pred_clean):.1%}")

## **Step 11: Addressing Imbalance with `scale_pos_weight`**

XGBoost's `scale_pos_weight` parameter increases the weight of the minority class during training. This serves the same purpose as `class_weight='balanced'` in Random Forest (Week 11). We set it to the ratio of Standard to High Value samples.

In [ ]:
ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight = {ratio:.1f}")

xgb_balanced = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                              scale_pos_weight=ratio, random_state=42,
                              eval_metric='logloss')
xgb_balanced.fit(X_train_clean, y_train)
y_pred_balanced = xgb_balanced.predict(X_test_clean)

In [ ]:
cm_naive = confusion_matrix(y_test, y_pred_clean)
cm_balanced = confusion_matrix(y_test, y_pred_balanced)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm_naive, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Standard', 'High Value'],
            yticklabels=['Standard', 'High Value'])
axes[0].set_title('Without scale_pos_weight')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(cm_balanced, annot=True, fmt='d', cmap='Greens', ax=axes[1],
            xticklabels=['Standard', 'High Value'],
            yticklabels=['Standard', 'High Value'])
axes[1].set_title('With scale_pos_weight')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

print("Without scale_pos_weight:")
print(classification_report(y_test, y_pred_clean,
                            target_names=['Standard', 'High Value'], zero_division=0))
print("With scale_pos_weight:")
print(classification_report(y_test, y_pred_balanced,
                            target_names=['Standard', 'High Value'], zero_division=0))

## **Step 12: Preventing Preprocessing Leakage with Pipelines**

Beyond removing leaky features, we should also ensure that preprocessing steps like scaling never see the test data. A `Pipeline` enforces this automatically: it fits the scaler only on training data and applies the learned transformation to the test set.

In [ ]:
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                            random_state=42, eval_metric='logloss'))
])
pipe.fit(X_train_clean, y_train)

print(f"Pipeline accuracy: {accuracy_score(y_test, pipe.predict(X_test_clean)):.1%}")
print(f"\nThe Pipeline ensures preprocessing never sees test data.")

## **Step 13: SHAP on the Clean Model**

With the leaky feature removed, we now have a trustworthy model. We apply SHAP to understand what the model has genuinely learned about housing prices.

In [ ]:
explainer_clean = shap.TreeExplainer(xgb_clean)
shap_values_clean = explainer_clean(X_test_clean)

print(f"Base value: {shap_values_clean.base_values[0]:.4f}")
print(f"SHAP values computed for {X_test_clean.shape[0]:,} samples across {X_test_clean.shape[1]} features.")

## **Step 14: SHAP Waterfall Plots — Local Interpretability**

The waterfall plot explains a **single prediction** by showing how each feature contributes, building from the base value to the final output.

- **Red bars** push the prediction higher (toward High Value).
- **Blue bars** push the prediction lower (toward Standard).

In [ ]:
probs = xgb_clean.predict_proba(X_test_clean)
hv_idx = probs[:, 1].argmax()

true_label = 'High Value' if y_test.iloc[hv_idx] == 1 else 'Standard'
pred_label = 'High Value' if xgb_clean.predict(X_test_clean.iloc[[hv_idx]])[0] == 1 else 'Standard'
print(f"Sample #{hv_idx}: True = {true_label}, Predicted = {pred_label}")

shap.plots.waterfall(shap_values_clean[hv_idx], max_display=10)

In [ ]:
std_idx = probs[:, 0].argmax()

true_label = 'High Value' if y_test.iloc[std_idx] == 1 else 'Standard'
pred_label = 'High Value' if xgb_clean.predict(X_test_clean.iloc[[std_idx]])[0] == 1 else 'Standard'
print(f"Sample #{std_idx}: True = {true_label}, Predicted = {pred_label}")

shap.plots.waterfall(shap_values_clean[std_idx], max_display=10)

## **Step 15: SHAP Feature Importance — Before and After**

Comparing SHAP importance before and after removing the leaky feature demonstrates the diagnostic power of SHAP analysis. In the leaky model, `TaxAssessment` drowned out every genuine signal. In the clean model, we see the true drivers of home value.

In [ ]:
mean_shap_clean = np.abs(shap_values_clean.values).mean(axis=0)
shap_imp_clean = pd.Series(mean_shap_clean, index=housing.feature_names)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

shap_imp_leaky.sort_values().plot(kind='barh', ax=ax1, color='crimson', edgecolor='black')
ax1.set_title('With TaxAssessment (Leaky)')
ax1.set_xlabel('Mean |SHAP Value|')

shap_imp_clean.sort_values().plot(kind='barh', ax=ax2, color='coral', edgecolor='black')
ax2.set_title('Without TaxAssessment (Clean)')
ax2.set_xlabel('Mean |SHAP Value|')

plt.tight_layout()
plt.show()

## **Step 16: SHAP Beeswarm Plot**

The beeswarm plot is the most information-dense SHAP visualization. Each dot represents a single sample in the test set:

- **Horizontal position** = SHAP value (how much that feature pushed the prediction).
- **Color** = the feature's actual value for that sample (red = high, blue = low).
- **Features** are sorted by overall importance (top = most impactful).

In [ ]:
shap.plots.beeswarm(shap_values_clean, max_display=10)

**Reading the beeswarm plot:**

For **MedInc** (median income): red dots (high income areas) cluster to the right (positive SHAP, pushing toward "High Value"), while blue dots (low income) cluster to the left (negative SHAP, pushing toward "Standard"). This makes intuitive sense — wealthier neighborhoods have more expensive homes.

For **Latitude**: certain latitudes correspond to high-cost coastal areas (e.g., the Bay Area, Southern California coast), while inland or northern latitudes tend toward lower values. The pattern may be non-linear, reflecting California's diverse geography.

## **Step 17: SHAP Dependence Plots**

Dependence plots show the exact relationship between a feature's value (x-axis) and its SHAP value (y-axis). Vertical spread at the same feature value indicates **interaction effects** — the feature's impact depends on other variables.

**Important:** SHAP tells us what the model learned — not what is true in the real world. A feature can be important to the model without having a causal relationship with the outcome. Always combine SHAP insights with domain expertise.

*([Cooper, 2021](https://www.aidancooper.co.uk/a-non-technical-guide-to-interpreting-shap-analyses/))*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

shap.plots.scatter(shap_values_clean[:, "MedInc"], ax=axes[0], show=False)
axes[0].set_title("MedInc (Median Income)")

shap.plots.scatter(shap_values_clean[:, "Latitude"], ax=axes[1], show=False)
axes[1].set_title("Latitude")

plt.tight_layout()
plt.show()